# **Day 1 and 2**

In [1]:
import random
import csv

transaction_col=["transaction_id","amount","merchant_category","country","hour_of_day","is_fraud","velocity_last_hour","distance_from_home_km","device_mismatch"]
merchant_arr=['electronics','grocery','restaurant',"travel"]
country_arr=['USA','UK','IND','BRA']

def generate_transaction(i):
    is_fraud = 1 if random.random() < 0.05 else 0  # calculate FIRST
    device_mismatch = random.random() < 0.80 if is_fraud == 1 else random.random() < 0.05

    transaction = {
        "transaction_id": "TXN-" + str(i).zfill(7),
        "amount": round(abs(random.gauss(800, 200)), 2) if is_fraud == 1 else round(abs(random.gauss(200, 100)), 2),
        "merchant_category": random.choice(merchant_arr),
        "country": random.choice(["NG", "RO", "UA"]) if is_fraud == 1 else random.choice(country_arr),
        "hour_of_day": random.randint(0, 4) if is_fraud == 1 else random.randint(8, 22),
        "is_fraud": is_fraud,
        "velocity_last_hour":random.randint(0,4) if is_fraud ==0 else random.randint(5,15),
        "distance_from_home_km":random.uniform(0,50) if is_fraud==0 else random.uniform(500,5000),
        "device_mismatch": device_mismatch
    }
    return transaction

fraud_count=0

with open("transactions.csv","w") as file:
  writer=csv.DictWriter(file,fieldnames=transaction_col)
  writer.writeheader()

  for i in range(10000):
   obj=generate_transaction(i)
  #  print(obj)
   writer.writerow(obj)
   if obj['is_fraud'] == 1:
     fraud_count=fraud_count+1

print(fraud_count)

511


Precision vs Recall

Imagine your model flags 50 transactions as fraud.

Precision answers — of those 50 flagged, how many were actually fraud? If only 10 were real fraud, precision = 10/50 = 20%. You're wrongly blocking 40 innocent customers.

Recall answers — of all the real fraud cases in your dataset, how many did you catch? If there were 15 real fraud cases and you caught 12, recall = 12/15 = 80%.

The tradeoff — if you want to catch every fraud case (high recall), you'll end up blocking some innocent people too. If you want to never bother innocent customers (high precision), you'll miss some fraud.

Capital One prioritizes recall — missing fraud costs them real money. Blocking a good customer just annoys them temporarily.

# **Day 3 - fine-tuning Llama 3.2**

In [2]:
!nvidia-smi

Tue Mar 17 02:36:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             30W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install transformers peft bitsandbytes trl datasets accelerate huggingface_hub torch

In [4]:
import transformers
import peft
import bitsandbytes
import trl
import datasets
import accelerate
import huggingface_hub

print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("trl:", trl.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("\nAll libraries loaded successfully!")

transformers: 5.0.0
peft: 0.18.1
bitsandbytes: 0.49.2
trl: 0.29.0
datasets: 4.0.0
accelerate: 1.13.0
huggingface_hub: 1.6.0

All libraries loaded successfully!


In [5]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_token'))

In [6]:
import csv
import json
import random

qa_pairs=[]

with open("transactions.csv","r") as file:
  reader=csv.DictReader(file)

  for row in reader:
    if(row['is_fraud']=="1"):
      prompt=f"Why was transaction {row['transaction_id']} flagged as fraud?"
      completion = (
                f"Transaction {row['transaction_id']} was flagged as fraud because: "
                f"Amount of ${row['amount']} which is unusually high, "
                f"originated from {row['country']} which is a high-risk country, "
                f"occurred at {row['hour_of_day']}:00 which is suspicious hours, "
                f"velocity of {row['velocity_last_hour']} transactions in last hour, "
                f"distance of {row['distance_from_home_km']}km from home, "
                f"device mismatch: {row['device_mismatch']}." )

      qa_pairs.append({"prompt":prompt,"completion":completion})

with open("finetune_qa.jsonl","w") as f:
  for pair in qa_pairs:
    f.write(json.dumps(pair)+"\n")

print(f"Generated {len(qa_pairs)} Q&A pairs")
print("\nSample:")
print(json.dumps(qa_pairs[0], indent=2))

Generated 511 Q&A pairs

Sample:
{
  "prompt": "Why was transaction TXN-0000004 flagged as fraud?",
  "completion": "Transaction TXN-0000004 was flagged as fraud because: Amount of $849.96 which is unusually high, originated from RO which is a high-risk country, occurred at 2:00 which is suspicious hours, velocity of 10 transactions in last hour, distance of 4844.27423138914km from home, device mismatch: False."
}


# **load TinyLlama**

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

#4bit quantization config
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded successfully!")
print(f"Model is on: {next(model.parameters()).device}")


Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading model with 4-bit quantization...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully!
Model is on: cuda:0


# **LoRA config + SFT training**

In [8]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import json

#step1
model = prepare_model_for_kbit_training(model)

#step2
lora_config=LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

#step3
model=get_peft_model(model,lora_config)
model.print_trainable_parameters()

#step4
data=[]
with open("finetune_qa.jsonl","r") as f:
  for line in f:
    pair=json.loads(line)
    data.append({"text": f"<|user|>\n{pair['prompt']}\n<|assistant|>\n{pair['completion']}"})

dataset=Dataset.from_list(data)
print(f"Dataset size: {len(dataset)}")

#step5
sft_config = SFTConfig(
    output_dir="./finshield-tinyllama",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    max_length=256,
    report_to="none"
)

# Step 6 — Train!
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
)

print("Starting training...")
trainer.train()
print("Training complete!")

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023
Dataset size: 511


Adding EOS to train dataset:   0%|          | 0/511 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/511 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/511 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting training...


Step,Training Loss
10,2.494432
20,1.706411
30,0.993813
40,0.637252
50,0.529647
60,0.500297
70,0.483407
80,0.472941
90,0.472344
100,0.470583


Training complete!


# **Save + Push to HuggingFace Hub**

In [9]:
# Save the LoRA adapter weights locally
model.save_pretrained("./finshield-tinyllama-adapter")
tokenizer.save_pretrained("./finshield-tinyllama-adapter")
print("Saved locally!")

# Push to HuggingFace Hub
model.push_to_hub("finshield-tinyllama-fraud", private=False)
tokenizer.push_to_hub("finshield-tinyllama-fraud", private=False)
print("Pushed to HuggingFace Hub!")

Saved locally!


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  28%|##7       |  632kB / 2.26MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Pushed to HuggingFace Hub!


# **Inference Test**

In [11]:
import torch

prompt = "<|user|>\nWhy was transaction TXN-0000004 flagged as fraud?\n<|assistant|>\n"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.3,        # lowered from 0.7 → 0.3 (less random)
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.3  # penalize repetitive garbage output
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

<|user|>
Why was transaction TXN-0000004 flagged as fraud?
<|assistant|>
Transactional meters and logs:<<<< <convert {<</code></precat < <divises such amount<<“<<<<<<<<< < < < <label<<<<<<|<<<commit_start < „<<<<<<<< <<<<<<<<<<<<<<> <recipolitan qu экс «мель «<<<<<|<<<<< < <{{’s increasing functional permissions<<<<<<<<<<<<<<<<<<<<<<| < < <<<<<<<<< < < <title<|number<


# **benchmarking**

In [15]:
import time
import torch

# Benchmark latency
prompt = "<|user|>\nWhy was transaction TXN-0000004 flagged as fraud?\n<|assistant|>\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Run 5 times and average
latencies = []
for i in range(5):
    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,        # greedy decoding for consistent benchmark
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True       # explicitly enable cache
        )
    end = time.time()
    latencies.append((end - start) * 1000)  # convert to ms
    print(f"Run {i+1}: {latencies[-1]:.0f}ms")

print(f"\n--- Benchmark Results ---")
print(f"Average latency: {sum(latencies)/len(latencies):.0f}ms")
print(f"Min latency: {min(latencies):.0f}ms")
print(f"Max latency: {max(latencies):.0f}ms")
print(f"Model: TinyLlama-1.1B + LoRA (4-bit quantized)")
print(f"Hardware: T4 GPU (15GB)")

Run 1: 14982ms
Run 2: 13820ms
Run 3: 14712ms
Run 4: 13483ms
Run 5: 14809ms

--- Benchmark Results ---
Average latency: 14361ms
Min latency: 13483ms
Max latency: 14982ms
Model: TinyLlama-1.1B + LoRA (4-bit quantized)
Hardware: T4 GPU (15GB)
